# Smart Waste Classification — Training Experiment

This notebook documents the reproducible experiment. The canonical implementation lives in
`src/waste_ml/`; the notebook calls the same modules so training and inference can also be
run from plain Python scripts.

Classes: `organic`, `plastic`, `paper`.
Architecture: MobileNetV2 (ImageNet weights) + GlobalAveragePooling2D + Dropout + Dense(3, softmax).

Run the cells from the repository root with the `uv` environment.

## 1. Environment

Requirements: Python 3.12, TensorFlow/Keras, Pillow, scikit-learn. Install with `uv sync`.

In [ ]:
import json
import platform
from pathlib import Path

import tensorflow as tf

print("Python:", platform.python_version())
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

## 2. Dataset

Source: https://huggingface.co/datasets/NeoAivara/Waste_Classification_data

Selected classes: `biological` → `organic`, `plastic`, `paper`.
The script deduplicates near-identical images before a stratified 70/15/15 split.

In [ ]:
# Regenerate only if dataset/ is missing or intentionally reset.
# !uv run python scripts/prepare_dataset.py --per-class 150 --seed 42

manifest = json.loads(Path("dataset/manifest.json").read_text())
print(json.dumps(manifest["counts"], indent=2))

## 3. Training

Two phases: frozen backbone feature extraction, then fine-tuning of the last 30
MobileNetV2 layers at a smaller learning rate. Callbacks: ModelCheckpoint, EarlyStopping,
ReduceLROnPlateau. Augmentation is applied to the training split only.

In [ ]:
# !uv run python -m waste_ml.training.train --epochs 20 --fine-tune-epochs 10 --seed 42

metadata = json.loads(Path("artifacts/model_metadata.json").read_text())
{
    "model_version": metadata["model_version"],
    "training_seconds": metadata["training_seconds"],
    "best_val_accuracy": metadata["best_val_accuracy"],
    "hyperparameters": metadata["hyperparameters"],
}

## 4. Evaluation

The held-out test split is used only here. Outputs: metrics.json, confusion matrix,
classification report, training history plot, and evaluation summary.

In [ ]:
# !uv run python -m waste_ml.training.evaluate

metrics = json.loads(Path("artifacts/evaluation/metrics.json").read_text())
print(f"Test accuracy: {metrics['accuracy'] * 100:.2f}%")
print("Confusion matrix:", metrics["confusion_matrix"])
print("Latency (ms):", metrics["latency_ms"]["end_to_end_ms"])

## 5. Inference sanity check

Load the saved `.keras` artifact once and classify one test image.

In [ ]:
from waste_ml.config import get_settings
from waste_ml.inference.predictor import WasteClassifier
from waste_ml.preprocessing import load_image

settings = get_settings()
classifier = WasteClassifier(
    settings.resolved_model_path,
    settings.resolved_model_metadata_path,
)
image_path = next(Path("dataset/test/plastic").glob("*.jpg"))
result = classifier.predict(load_image(image_path.read_bytes()))
result